# Advanced: Multi-Strategy Consistent Mapping Pseudonymization with PAMOLA.CORE

**Goal**: Master all 4 consistent mapping pseudonymization strategies using operation.execute()

**What you'll learn:**
- **Strategy 1**: Format-preserving pseudonymization with regex patterns
- **Strategy 2**: Random character pseudonymization with pools
- **Strategy 3**: Fixed-length pseudonymization for uniformity
- **Strategy 4**: Risk-based conditional pseudonymization with k-anonymity
- Calculate advanced privacy metrics
- Analyze pseudonymization effectiveness
- Production deployment patterns

**Prerequisites:**
- Completed the simple notebook
- Understanding of privacy concepts
- Familiarity with operation.execute() pattern
- Python 3.8+
- PAMOLA.CORE installed (auto-installs if needed)

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── anonymization/pseudonymization/
        └── 02_mapping_pseudonymization_advanced.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import json
import sys
import os
from pathlib import Path
from datetime import datetime
import time
from IPython.display import Image, display

print("🔍 Detecting PAMOLA installation...\n")

# Get current notebook location
current_dir = Path.cwd()
print(f"📍 Notebook location: {current_dir}")

# Navigate up to find project root (where pamola_core exists)
project_root = current_dir
pamola_found = False

# Go up maximum 6 levels to find pamola_core
for level in range(6):
    if (project_root / 'pamola_core').exists():
        print(f"✓ Found pamola_core at level {level}: {project_root}")
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break
    project_root = project_root.parent

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print(f"   Searched up to: {project_root}")
    print(f"   Current location: {current_dir}")
    print("\n📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print("   Repository: https://github.com/DGT-Network/PAMOLA.git")
        print("   Branch: pre-epic3")
        print("\n⚠️  IMPORTANT: Please restart your kernel before continuing!")
        print("   (Kernel → Restart Kernel)")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        print("\n💡 Manual installation:")
        print("   pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed."
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Add to Python path (if found locally)
if pamola_found:
    print(f"✓ Added to sys.path\n")

# Import PAMOLA modules
print("📦 Importing PAMOLA modules...")
try:
    from pamola_core.anonymization.pseudonymization.mapping_op import (
        ConsistentMappingPseudonymizationOperation
    )
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv

    print("✅ All imports successful!\n")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Restart your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Check Python version: python --version (requires 3.8+)")
    print("   4. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Complex Dataset

**How to use:**
- Run to load from `examples/data_examples/sample.csv`
- Auto-generates 1000-record dataset if file missing
- Review data structure before configuring strategies

**What you'll see:**
- Dataset overview (1000 records, 8 columns)
- First 5 sample records
- Column statistics (types, ranges, unique counts)
- Sensitive fields: ssn, phone, email, credit_card, account_number
- K-risk scores (1-10) for risk-based strategy testing

**Note:** Generated data automatically saved to file for future runs. Large dataset optimized for strategy comparison

In [ ]:
# Try to load from file first (in examples directory)
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'sample.csv'
print(f"📂 Looking for data at: {data_path}\n")

if data_path.exists():
    print(f"📂 Loading data from: {data_path}")
    df = read_csv(data_path)
    print(f"✓ Loaded {len(df)} records from file")
else:
    print("📊 Generating synthetic dataset (1000 records)...\n")
    
    np.random.seed(42)
    n_records = 1000
    
    # Generate diverse sensitive data
    df = pd.DataFrame({
        'record_id': range(1, n_records + 1),
        
        # SSN with format XXX-XX-XXXX
        'ssn': [f"{np.random.randint(100,999):03d}-{np.random.randint(10,99):02d}-{np.random.randint(1000,9999):04d}" 
                for _ in range(n_records)],
        
        # Phone with format (XXX) XXX-XXXX
        'phone': [f"({np.random.randint(200,999):03d}) {np.random.randint(200,999):03d}-{np.random.randint(1000,9999):04d}" 
                  for _ in range(n_records)],
        
        # Email addresses
        'email': [f"user{i}@domain{np.random.randint(1,10)}.com" for i in range(1, n_records + 1)],
        
        # Credit card with format XXXX-XXXX-XXXX-XXXX
        'credit_card': [f"{np.random.randint(1000,9999):04d}-{np.random.randint(1000,9999):04d}-{np.random.randint(1000,9999):04d}-{np.random.randint(1000,9999):04d}" 
                        for _ in range(n_records)],
        
        # Account numbers (numeric)
        'account_number': np.random.randint(100000000, 999999999, n_records),
        
        # Department for grouping
        'department': np.random.choice(['Engineering', 'Sales', 'HR', 'Marketing', 'Finance', 'Operations'], n_records),
        
        # K-anonymity risk scores (1-10, lower is more vulnerable)
        'k_risk_score': np.random.randint(1, 11, n_records),
        
        # Salary for context
        'salary': np.random.randint(50000, 150000, n_records)
    })
    
    # Save for future use (with error handling)
    try:
        data_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(data_path, index=False)
        print(f"✓ Generated and saved to: {data_path}")
    except PermissionError:
        print(f"⚠️  Cannot save to {data_path} (file may be open)")
        print(f"   Dataset generated in memory only")

print(f"\n📊 Dataset Overview:")
print("=" * 80)
print(f"  Records: {len(df):,}")
print(f"  Columns: {', '.join(df.columns)}")

print(f"\n🔍 Sample Records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:20s} ({dtype_str:10s}): {df[col].nunique()} unique values")
    else:
        print(f"  {col:20s} ({dtype_str:10s}): min={df[col].min()}, max={df[col].max()}")

print(f"\n💡 Perfect dataset for testing all 4 pseudonymization strategies!")

## Step 3: Setup Shared Environment

**How to use:**
1. **CUSTOMIZE FIELD_CONFIG** dictionary with your target fields
2. Run to validate all fields exist in dataset
3. Setup shared components for all strategies

**What you'll see:**
- Field validation (✓ marks for each strategy field with unique counts)
- Task directory created
- TaskReporter and DataSource initialized (✓)
- Success message (✅ Shared environment ready!)

**Field configuration:**
```python
FIELD_CONFIG = {
    "strategy1_field": "email",           # Format-preserving
    "strategy2_field": "email",         # Random pseudonymization
    "strategy3_field": "email",         # Fixed-length
    "strategy4_field": "email"    # Risk-based
}
```

**Note:** All fields must exist in dataset. Shared environment used across all 4 strategies

In [ ]:
# Field configuration - CUSTOMIZE THESE!
FIELD_CONFIG = {
    "strategy1_field": "email",           # Format-preserving pseudonymization
    "strategy2_field": "email",         # Random character pseudonymization
    "strategy3_field": "email",         # Fixed-length pseudonymization
    "strategy4_field": "email"    # Risk-based conditional pseudonymization
}

# Validate all fields exist in dataset
print("📋 Validating field configuration...\n")
for strategy, field_name in FIELD_CONFIG.items():
    if field_name not in df.columns:
        raise ValueError(
            f"❌ Field '{field_name}' for {strategy} not found!\n"
            f"Available columns: {', '.join(df.columns)}\n"
            f"Please update FIELD_CONFIG"
        )
    print(f"  ✓ {strategy:20s}: '{field_name}' ({df[field_name].nunique()} unique values)")

# Configuration helper (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset"
    }

# Create task directory (in examples/data_examples)
examples_dir = project_root / 'examples'
task_dir = examples_dir / 'data_examples' / 'mapping_advanced_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="advanced_pseudo_001",
    task_type="multi_strategy_pseudonymization",
    description="Multi-strategy consistent mapping pseudonymization processing",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

# Get config
kwargs = create_config_kwargs()
print(f"✓ Config kwargs ready")

# Create DataSource
data_source = DataSource(
    dataframes={"main_dataset": df}
)
print("✓ DataSource created")

print(f"\n✅ Shared environment ready for all strategies!")

## STRATEGY 1: UUID Pseudonyms (Default)

**How to use:**
- Run to pseudonymize using RFC 4122 UUID pseudonyms

**Note:** Output field: `{field_name}_uuid`. Reversible via encrypted mapping.


In [ ]:
print("\n" + "=" * 80)
print("STRATEGY 1: UUID PSEUDONYMS (DEFAULT)")
print("=" * 80)

# Setup progress tracker
tracker_s1 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 1: UUID Pseudonyms",
    unit="steps",
    track_memory=True,
    level=0,
)

field_s1 = FIELD_CONFIG['strategy1_field']
op_s1 = ConsistentMappingPseudonymizationOperation(
    field_name=field_s1,
    mode='ENRICH',
    column_prefix='_',
    output_field_name=f"{field_s1}_uuid",
    mapping_encryption_key='ab' * 32,
    pseudonym_type='uuid',
    mapping_format='csv',
    create_if_not_exists=True,
    backup_on_update=True,
    # Processing settings
    use_dask=True,
    npartitions=2,
    use_cache=False,

    generate_visualization=True,
    save_output=True,
)

import time
start_time = time.time()

result_s1 = op_s1.execute(
    data_source=DataSource(dataframes={"main_dataset": df.copy()}),
    task_dir=task_dir / 'strategy1_uuid',
    reporter=task_reporter,
    progress_tracker=tracker_s1,
    dataset_name="main_dataset",
)

elapsed_s1 = time.time() - start_time
print(f"Strategy 1 elapsed: {elapsed_s1:.2f}s")
print(f"Strategy 1 completed - Status: {result_s1.status}")
output_files_s1 = list((task_dir / 'strategy1_uuid' / 'output').glob('*.csv'))
if output_files_s1:
    result_df_s1 = pd.read_csv(output_files_s1[0])
    output_field_s1 = f"{field_s1}_uuid"
    print("   Preview:")
    for i in range(min(3, len(result_df_s1))):
        print(f"      {result_df_s1[field_s1].iloc[i]} -> {result_df_s1[output_field_s1].iloc[i]}")


## STRATEGY 2: Sequential Pseudonyms with Prefix

**How to use:**
- Run to pseudonymize with monotonically increasing integer pseudonyms
- Efficient for large datasets; enumerable

**Note:** Output field: `{field_name}_seq`. Pseudonyms like `CUST_1`, `CUST_2`, ...


In [ ]:
print("\n" + "=" * 80)
print("STRATEGY 2: SEQUENTIAL PSEUDONYMS WITH PREFIX")
print("=" * 80)

# Setup progress tracker
tracker_s2 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 2: Sequential Pseudonyms",
    unit="steps",
    track_memory=True,
    level=0,
)

field_s2 = FIELD_CONFIG['strategy2_field']
op_s2 = ConsistentMappingPseudonymizationOperation(
    field_name=field_s2,
    mode='ENRICH',
    column_prefix='_',
    output_field_name=f"{field_s2}_seq",
    mapping_encryption_key='cd' * 32,
    pseudonym_type='sequential',
    pseudonym_prefix='CUST_',
    mapping_format='csv',
    # Processing settings
    use_dask=True,
    npartitions=2,
    use_cache=False,

    generate_visualization=True,
    save_output=True,
)

import time
start_time = time.time()

result_s2 = op_s2.execute(
    data_source=DataSource(dataframes={"main_dataset": df.copy()}),
    task_dir=task_dir / 'strategy2_seq',
    reporter=task_reporter,
    progress_tracker=tracker_s2,
    dataset_name="main_dataset",
)

elapsed_s2 = time.time() - start_time
print(f"Strategy 2 elapsed: {elapsed_s2:.2f}s")
print(f"Strategy 2 completed - Status: {result_s2.status}")
output_files_s2 = list((task_dir / 'strategy2_seq' / 'output').glob('*.csv'))
if output_files_s2:
    result_df_s2 = pd.read_csv(output_files_s2[0])
    output_field_s2 = f"{field_s2}_seq"
    print("   Preview:")
    for i in range(min(3, len(result_df_s2))):
        print(f"      {result_df_s2[field_s2].iloc[i]} -> {result_df_s2[output_field_s2].iloc[i]}")


## STRATEGY 3: Random-String Pseudonyms

**How to use:**
- Run to pseudonymize with random alphanumeric strings of fixed length

**Note:** Output field: `{field_name}_rand`.


In [ ]:
print("\n" + "=" * 80)
print("STRATEGY 3: RANDOM-STRING PSEUDONYMS")
print("=" * 80)

# Setup progress tracker
tracker_s3 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 3: Random-String Pseudonyms",
    unit="steps",
    track_memory=True,
    level=0,
)

field_s3 = FIELD_CONFIG['strategy3_field']
op_s3 = ConsistentMappingPseudonymizationOperation(
    field_name=field_s3,
    mode='ENRICH',
    column_prefix='_',
    output_field_name=f"{field_s3}_rand",
    mapping_encryption_key='ef' * 32,
    pseudonym_type='random_string',
    pseudonym_length=16,
    mapping_format='json',
    # Processing settings
    use_dask=True,
    npartitions=2,
    use_cache=False,

    generate_visualization=True,
    save_output=True,
)

import time
start_time = time.time()

result_s3 = op_s3.execute(
    data_source=DataSource(dataframes={"main_dataset": df.copy()}),
    task_dir=task_dir / 'strategy3_rand',
    reporter=task_reporter,
    progress_tracker=tracker_s3,
    dataset_name="main_dataset",
)

elapsed_s3 = time.time() - start_time
print(f"Strategy 3 elapsed: {elapsed_s3:.2f}s")
print(f"Strategy 3 completed - Status: {result_s3.status}")
output_files_s3 = list((task_dir / 'strategy3_rand' / 'output').glob('*.csv'))
if output_files_s3:
    result_df_s3 = pd.read_csv(output_files_s3[0])
    output_field_s3 = f"{field_s3}_rand"
    print("   Preview:")
    for i in range(min(3, len(result_df_s3))):
        print(f"      {result_df_s3[field_s3].iloc[i]} -> {result_df_s3[output_field_s3].iloc[i]}")


## STRATEGY 4: Compound Multi-Field Mapping

**How to use:**
- Run to map multiple fields together as a single compound identifier

**Note:** Output field: `{field_name}_compound`.


In [ ]:
print("\n" + "=" * 80)
print("STRATEGY 4: COMPOUND MULTI-FIELD MAPPING")
print("=" * 80)

# Setup progress tracker
tracker_s4 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 4: Compound Multi-Field Mapping",
    unit="steps",
    track_memory=True,
    level=0,
)

field_s4 = FIELD_CONFIG['strategy4_field']
additional = [FIELD_CONFIG.get('strategy1_field', field_s4)] if FIELD_CONFIG.get('strategy1_field') != field_s4 else []

op_s4 = ConsistentMappingPseudonymizationOperation(
    field_name=field_s4,
    additional_fields=additional,
    mode='ENRICH',
    column_prefix='_',
    output_field_name=f"{field_s4}_compound",
    mapping_encryption_key='01' * 32,
    pseudonym_type='uuid',
    compound_mode=bool(additional),
    compound_separator='|',
    compound_null_handling='skip',
    mapping_format='csv',
    # Processing settings
    use_dask=True,
    npartitions=2,
    use_cache=False,

    generate_visualization=True,
    save_output=True,
)

import time
start_time = time.time()

result_s4 = op_s4.execute(
    data_source=DataSource(dataframes={"main_dataset": df.copy()}),
    task_dir=task_dir / 'strategy4_compound',
    reporter=task_reporter,
    progress_tracker=tracker_s4,
    dataset_name="main_dataset",
)

elapsed_s4 = time.time() - start_time
print(f"Strategy 4 elapsed: {elapsed_s4:.2f}s")
print(f"Strategy 4 completed - Status: {result_s4.status}")
output_files_s4 = list((task_dir / 'strategy4_compound' / 'output').glob('*.csv'))
if output_files_s4:
    result_df_s4 = pd.read_csv(output_files_s4[0])
    output_field_s4 = f"{field_s4}_compound"
    print(f"   Compound mapping used: {bool(additional)}")
    print(f"   Additional fields: {additional}")
    print("   Preview:")
    for i in range(min(3, len(result_df_s4))):
        print(f"      {result_df_s4[field_s4].iloc[i]} -> {result_df_s4[output_field_s4].iloc[i]}")


## Step 4: Strategy Comparison

**How to use:**
- Run AFTER all 4 strategies complete successfully
- Review execution times and pseudonymization coverage
- Identify best strategy for your use case

**What you'll see:**
- Execution time comparison (all 4 strategies)
- Pseudonymization coverage percentages per strategy
- Total processing time
- Strategy recommendations by use case

**Note:** Helps choose optimal strategy based on requirements. Risk-based strategy typically has lower coverage but better utility

In [ ]:
print("\n" + "=" * 80)
print("📊 STRATEGY COMPARISON")
print("=" * 80 + "\n")

print("⏱️  Execution Time:")
print(f"  Strategy 1 (UUID):                   {elapsed_s1:6.2f}s")
print(f"  Strategy 2 (Sequential):                {elapsed_s2:6.2f}s")
print(f"  Strategy 3 (Random-String):         {elapsed_s3:6.2f}s")
print(f"  Strategy 4 (Compound Multi-Field):  {elapsed_s4:6.2f}s")
print(f"  Total:                          {elapsed_s1 + elapsed_s2 + elapsed_s3 + elapsed_s4:6.2f}s")

print(f"\n📊 Pseudonymization Coverage:")
if 'result_df_s1' in locals():
    field_s1 = FIELD_CONFIG["strategy1_field"]
    output_field_s1 = f"{field_s1}_uuid"
    if output_field_s1 in result_df_s1.columns:
        pseudonymized_count = (result_df_s1[field_s1] != result_df_s1[output_field_s1]).sum()
        coverage_s1 = (pseudonymized_count / len(result_df_s1)) * 100
        print(f"  Strategy 1 (UUID):               {coverage_s1:.1f}% of records pseudonymized")

if 'result_df_s2' in locals():
    field_s2 = FIELD_CONFIG["strategy2_field"]
    output_field_s2 = f"{field_s2}_seq"
    if output_field_s2 in result_df_s2.columns:
        pseudonymized_count = (result_df_s2[field_s2] != result_df_s2[output_field_s2]).sum()
        coverage_s2 = (pseudonymized_count / len(result_df_s2)) * 100
        print(f"  Strategy 2 (Sequential):         {coverage_s2:.1f}% of records pseudonymized")

if 'result_df_s3' in locals():
    field_s3 = FIELD_CONFIG["strategy3_field"]
    output_field_s3 = f"{field_s3}_rand"
    if output_field_s3 in result_df_s3.columns:
        pseudonymized_count = (result_df_s3[field_s3] != result_df_s3[output_field_s3]).sum()
        coverage_s3 = (pseudonymized_count / len(result_df_s3)) * 100
        print(f"  Strategy 3 (Random-String):      {coverage_s3:.1f}% of records pseudonymized")

if 'result_df_s4' in locals():
    field_s4 = FIELD_CONFIG["strategy4_field"]
    output_field_s4 = f"{field_s4}_compound"
    if output_field_s4 in result_df_s4.columns:
        pseudonymized_count = (result_df_s4[field_s4] != result_df_s4[output_field_s4]).sum()
        coverage_s4 = (pseudonymized_count / len(result_df_s4)) * 100
        print(f"  Strategy 4 (Compound):           {coverage_s4:.1f}% of records pseudonymized")

print(f"\n💡 Strategy Recommendations:")
print(f"  • UUID: Use when globally unique, opaque identifiers are required")
print(f"  • Sequential: Use when enumerable, compact pseudonyms are required")
print(f"  • Random-String: Use when maximum unpredictability is needed")
print(f"  • Compound Multi-Field: Use when joint identifiers must map as a single unit")

## Step 5: Detailed Artifact Review (All Strategies)

**How to use:**
- Run to review all strategy artifacts in one place
- Focus on key validation points for each strategy

**What you'll see (per strategy):**
1. **Metrics JSON**: Effectiveness ratio, performance, pseudonymization stats
2. **Visualizations**: First 2 charts displayed inline

**Validate:**
- ✅ Unique value reduction (effectiveness ratio)
- ✅ Processing speed (records/second)
- ✅ Pseudonymization coverage percentage
- ✅ Format preservation counts (Strategy 1)
- ✅ Risk-based pseudonymization counts (Strategy 4)

**Note:** Only newest files shown per strategy. Excludes older test runs automatically

In [ ]:
# Review all strategies
strategy_dirs = [
    ('strategy1_uuid', 'Strategy 1: UUID Pseudonyms'),
    ('strategy2_seq', 'Strategy 2: Sequential Pseudonyms'),
    ('strategy3_rand', 'Strategy 3: Random-String Pseudonyms'),
    ('strategy4_compound', 'Strategy 4: Compound Multi-Field Mapping')
]

for dir_name, title in strategy_dirs:
    strategy_dir = task_dir / dir_name
    if not strategy_dir.exists():
        continue

    print("\n" + "=" * 80)
    print(f"📊 {title}")
    print("=" * 80)

    # 1. Metrics (NEWEST - with filtering)
    metrics_dir = strategy_dir / 'metrics'
    if metrics_dir.exists():
        metrics_files = list(metrics_dir.glob('*.json'))

        if metrics_files:
            # 1) Exclude data_types files
            filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

            if filtered:
                target_files = filtered
                print(f"✓ Found {len(filtered)} filtered metrics file(s) (excluded data_types)")
            else:
                target_files = metrics_files
                print(f"⚠ No filtered metrics found → fallback to ALL {len(metrics_files)} file(s)")

            # 2) Pick latest
            target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
            latest_metrics_file = target_files[0]

            print(f"📄 Loading LATEST: {latest_metrics_file.name}")
            mtime = datetime.fromtimestamp(latest_metrics_file.stat().st_mtime)
            print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
            print(f"   Size: {latest_metrics_file.stat().st_size / 1024:.1f} KB\n")
            print("=" * 80)

            try:
                with open(latest_metrics_file, 'r') as f:
                    data = json.load(f)
                    metrics = data.get('metrics', {})

                if 'effectiveness' in metrics:
                    eff = metrics['effectiveness']
                    print(f"   Original: {eff.get('original_unique')} → Pseudonymized: {eff.get('anonymized_unique')}")
                    print(f"   Reduction: {eff.get('effectiveness_ratio', 0):.2%}")

                if 'performance' in metrics:
                    perf = metrics['performance']
                    print(f"   Duration: {perf.get('duration_seconds', 0):.4f}s")
                    print(f"   Records/sec: {perf.get('records_per_second', 0):.2f}")

            except Exception as e:
                print(f"   ⚠️  Error: {e}")

    # 2. Visualizations (NEWEST BATCH)
    viz_dir = strategy_dir / 'visualizations'
    if viz_dir.exists():
        viz_files = sorted(
            list(viz_dir.glob('*.png')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )

        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            latest_batch = [
                f for f in viz_files
                if abs(f.stat().st_mtime - latest_time) < 10
            ]

            print(f"\n📊 VISUALIZATIONS: {len(latest_batch)} files")
            for viz in latest_batch[:2]:
                print(f"   📈 {viz.stem.replace('_', ' ').title()}")
                try:
                    display(Image(filename=str(viz)))
                except:
                    print(f"      (Display error)")

## Step 6: Calculate Privacy Metrics

**How to use:**
- Run AFTER all 4 strategies complete successfully
- Compare k-anonymity before and after pseudonymization
- Verify privacy improvements

**What you'll see:**
- Original data k-anonymity (min k, avg k, vulnerable groups)
- After pseudonymization k-anonymity (Strategies 1 & 2 combined)
- Improvement ratios (e.g., "5.2x better")
- Final k-anonymity level achieved

**Note:** Higher k values = better privacy protection. Combined strategies typically achieve k > 100

In [ ]:
print("\n" + "=" * 80)
print("🔒 PRIVACY METRICS")
print("=" * 80 + "\n")

def calculate_k_anonymity(df, quasi_identifiers):
    """Calculate k-anonymity metrics"""
    existing_qis = [q for q in quasi_identifiers if q in df.columns]
    if not existing_qis:
        return None

    group_sizes = df.groupby(existing_qis).size()
    return {
        'min_k': int(group_sizes.min()),
        'avg_k': float(group_sizes.mean()),
        'vulnerable_groups': int((group_sizes < 5).sum())
    }

try:
    field_s1 = FIELD_CONFIG["strategy1_field"]

    # Baseline: original field + department as quasi-identifiers
    quasi_ids_orig = [field_s1, 'department']
    k_orig = calculate_k_anonymity(df, quasi_ids_orig)
    print(f"📊 ORIGINAL DATA (quasi-IDs: {quasi_ids_orig}):")
    print(f"   Min k: {k_orig['min_k']}")
    print(f"   Avg k: {k_orig['avg_k']:.2f}")
    print(f"   Vulnerable groups (k<5): {k_orig['vulnerable_groups']}")

    # After pseudonymization: use Strategy 1 UUID output + department
    if 'result_df_s1' in locals() and result_df_s1 is not None:
        pseudonymized_field_s1 = f"{field_s1}_uuid"

        if pseudonymized_field_s1 in result_df_s1.columns and 'department' in result_df_s1.columns:
            quasi_ids_final = [pseudonymized_field_s1, 'department']
            k_final = calculate_k_anonymity(result_df_s1, quasi_ids_final)

            print(f"\n✨ AFTER PSEUDONYMIZATION (quasi-IDs: {quasi_ids_final}):")
            print(f"   Min k: {k_final['min_k']} ({k_final['min_k']/max(k_orig['min_k'],1):.1f}x)")
            print(f"   Avg k: {k_final['avg_k']:.2f} ({k_final['avg_k']/max(k_orig['avg_k'],1e-9):.1f}x)")
            print(f"   Vulnerable groups (k<5): {k_final['vulnerable_groups']}")

            print(f"\n🎯 Dataset is {k_final['min_k']}-anonymous after pseudonymization")
        else:
            print(f"\n⚠️  Required columns not found in result_df_s1")
            print(f"   Expected: {pseudonymized_field_s1}, department")
            print(f"   Available: {result_df_s1.columns.tolist()}")
    else:
        print("\n⚠️  Pseudonymization not completed yet - run all strategies first!")

except NameError:
    print("⚠️  FIELD_CONFIG not defined!")
    print("   Please run 'Step 3: Setup Shared Environment' cell first")
    print("   Then run all strategy cells before calculating privacy metrics")

## Step 7: Export Final Data

**How to use:**
- Run to export all strategy results
- Each strategy saved to separate folder
- Review export summary for file locations

**What you'll see:**
- Export confirmation per strategy with file paths
- File statistics (rows, columns)
- First 5 rows preview per strategy
- Pseudonymization statistics summary per strategy
- Total processing time

**Output structure:**
```
advanced_output/
├── strategy1/strategy1_data.csv
├── strategy2/strategy2_data.csv
├── strategy3/strategy3_data.csv
└── strategy4/strategy4_data.csv
```

**Note:** Check console for permission errors. Files may be locked if open in another program

In [ ]:
import os
from pathlib import Path

print("=" * 80)
print("📦 EXPORTING FINAL DATA FROM ALL STRATEGIES")
print("=" * 80)

# Get field config
try:
    field_s1 = FIELD_CONFIG["strategy1_field"]
    field_s2 = FIELD_CONFIG["strategy2_field"]
    field_s3 = FIELD_CONFIG["strategy3_field"]
    field_s4 = FIELD_CONFIG["strategy4_field"]
except NameError:
    print("❌ FIELD_CONFIG not defined! Run Step 3 first.")
    raise

# Output column names (must match each operation's output_field_name)
output_col_s1 = f"{field_s1}_uuid"
output_col_s2 = f"{field_s2}_seq"
output_col_s3 = f"{field_s3}_rand"
output_col_s4 = f"{field_s4}_compound"

# Base export directory
export_base_dir = task_dir
task_dir.mkdir(parents=True, exist_ok=True)

print(f"\n📂 Export base directory: {task_dir}\n")

# ============================================================================
# STRATEGY 1: UUID Pseudonyms
# ============================================================================
if 'result_df_s1' in locals() and result_df_s1 is not None:
    print("=" * 80)
    print("📊 STRATEGY 1: UUID PSEUDONYMS")
    print("=" * 80)

    s1_dir = export_base_dir / 'strategy1'
    s1_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n📋 Available columns:")
    print(f"  {result_df_s1.columns.tolist()}")

    if output_col_s1 in result_df_s1.columns:
        export_cols_s1 = ['record_id', field_s1, output_col_s1, 'department', 'salary']
        existing_cols_s1 = [col for col in export_cols_s1 if col in result_df_s1.columns]

        final_df_s1 = result_df_s1[existing_cols_s1].copy()

        output_path_s1 = s1_dir / 'strategy1_data.csv'
        try:
            final_df_s1.to_csv(output_path_s1, index=False)
            print(f"\n✅ Saved: {output_path_s1}")
            print(f"   Columns: {len(final_df_s1.columns)}")
            print(f"   Rows: {len(final_df_s1):,}")
        except PermissionError:
            print(f"\n⚠️  Cannot save (file may be open): {output_path_s1}")

        print(f"\n📊 Preview (first 5 rows):")
        print(final_df_s1.head())

        print(f"\n📈 Pseudonymization Statistics:")
        print(f"   Original unique values: {result_df_s1[field_s1].nunique()}")
        print(f"   Pseudonymized unique values: {result_df_s1[output_col_s1].nunique()}")
        reduction = (1 - result_df_s1[output_col_s1].nunique() / max(result_df_s1[field_s1].nunique(), 1)) * 100
        print(f"   Reduction: {reduction:.1f}%")
    else:
        print(f"\n❌ Column '{output_col_s1}' not found in result_df_s1")
        print(f"   Available columns: {result_df_s1.columns.tolist()}")
else:
    print("\n⚠️  Strategy 1 data not available (result_df_s1 not found)")

# ============================================================================
# STRATEGY 2: Sequential Pseudonyms
# ============================================================================
if 'result_df_s2' in locals() and result_df_s2 is not None:
    print("\n" + "=" * 80)
    print("📊 STRATEGY 2: SEQUENTIAL PSEUDONYMS")
    print("=" * 80)

    s2_dir = export_base_dir / 'strategy2'
    s2_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n📋 Available columns:")
    print(f"  {result_df_s2.columns.tolist()}")

    if output_col_s2 in result_df_s2.columns:
        export_cols_s2 = ['record_id', field_s2, output_col_s2, 'department', 'salary']
        existing_cols_s2 = [col for col in export_cols_s2 if col in result_df_s2.columns]

        final_df_s2 = result_df_s2[existing_cols_s2].copy()

        output_path_s2 = s2_dir / 'strategy2_data.csv'
        try:
            final_df_s2.to_csv(output_path_s2, index=False)
            print(f"\n✅ Saved: {output_path_s2}")
            print(f"   Columns: {len(final_df_s2.columns)}")
            print(f"   Rows: {len(final_df_s2):,}")
        except PermissionError:
            print(f"\n⚠️  Cannot save (file may be open): {output_path_s2}")

        print(f"\n📊 Preview (first 5 rows):")
        print(final_df_s2.head())

        print(f"\n📈 Pseudonymization Statistics:")
        print(f"   Field: {field_s2}")
        print(f"   Original unique values: {result_df_s2[field_s2].nunique()}")
        print(f"   Pseudonymized unique values: {result_df_s2[output_col_s2].nunique()}")
        reduction = (1 - result_df_s2[output_col_s2].nunique() / max(result_df_s2[field_s2].nunique(), 1)) * 100
        print(f"   Reduction: {reduction:.1f}%")
    else:
        print(f"\n❌ Column '{output_col_s2}' not found")
else:
    print("\n⚠️  Strategy 2 data not available")

# ============================================================================
# STRATEGY 3: Random-String Pseudonyms
# ============================================================================
if 'result_df_s3' in locals() and result_df_s3 is not None:
    print("\n" + "=" * 80)
    print("📊 STRATEGY 3: RANDOM-STRING PSEUDONYMS")
    print("=" * 80)

    s3_dir = export_base_dir / 'strategy3'
    s3_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n📋 Available columns:")
    print(f"  {result_df_s3.columns.tolist()}")

    if output_col_s3 in result_df_s3.columns:
        export_cols_s3 = ['record_id', field_s3, output_col_s3, 'department', 'salary']
        existing_cols_s3 = [col for col in export_cols_s3 if col in result_df_s3.columns]

        final_df_s3 = result_df_s3[existing_cols_s3].copy()

        output_path_s3 = s3_dir / 'strategy3_data.csv'
        try:
            final_df_s3.to_csv(output_path_s3, index=False)
            print(f"\n✅ Saved: {output_path_s3}")
            print(f"   Columns: {len(final_df_s3.columns)}")
            print(f"   Rows: {len(final_df_s3):,}")
        except PermissionError:
            print(f"\n⚠️  Cannot save (file may be open): {output_path_s3}")

        print(f"\n📊 Preview (first 5 rows):")
        print(final_df_s3.head())

        print(f"\n📈 Pseudonymization Statistics:")
        print(f"   Field: {field_s3}")
        print(f"   Original unique values: {result_df_s3[field_s3].nunique()}")
        print(f"   Pseudonymized unique values: {result_df_s3[output_col_s3].nunique()}")
        reduction = (1 - result_df_s3[output_col_s3].nunique() / max(result_df_s3[field_s3].nunique(), 1)) * 100
        print(f"   Reduction: {reduction:.1f}%")
    else:
        print(f"\n❌ Column '{output_col_s3}' not found")
else:
    print("\n⚠️  Strategy 3 data not available")

# ============================================================================
# STRATEGY 4: Compound Multi-Field Mapping
# ============================================================================
if 'result_df_s4' in locals() and result_df_s4 is not None:
    print("\n" + "=" * 80)
    print("📊 STRATEGY 4: COMPOUND MULTI-FIELD MAPPING")
    print("=" * 80)

    s4_dir = export_base_dir / 'strategy4'
    s4_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n📋 Available columns:")
    print(f"  {result_df_s4.columns.tolist()}")

    if output_col_s4 in result_df_s4.columns:
        export_cols_s4 = ['record_id', field_s4, output_col_s4, 'department', 'salary']
        existing_cols_s4 = [c for c in export_cols_s4 if c in result_df_s4.columns]
        final_df_s4 = result_df_s4[existing_cols_s4].copy()

        output_path_s4 = s4_dir / 'strategy4_data.csv'
        try:
            final_df_s4.to_csv(output_path_s4, index=False)
            print(f"\n✅ Saved: {output_path_s4}")
            print(f"   Columns: {len(final_df_s4.columns)}")
            print(f"   Rows: {len(final_df_s4):,}")
        except PermissionError:
            print(f"\n⚠️  Cannot save (file may be open): {output_path_s4}")

        print(f"\n📊 Preview (first 5 rows):")
        print(final_df_s4.head())

        print(f"\n📈 Pseudonymization Statistics:")
        print(f"   Field: {field_s4}")

        orig_series = final_df_s4[field_s4]
        pseudonymized_series = final_df_s4[output_col_s4]
        pseudonymized = (orig_series != pseudonymized_series).sum()

        print(f"   Total records: {len(final_df_s4)}")
        print(f"   Actually pseudonymized (field changed): {pseudonymized}")
        print(f"   Preserved: {len(final_df_s4) - pseudonymized}")
        print(f"   Original unique values: {result_df_s4[field_s4].nunique()}")
        print(f"   Pseudonymized unique values: {result_df_s4[output_col_s4].nunique()}")
    else:
        print(f"\n❌ Column '{output_col_s4}' not found in result_df_s4")
else:
    print("\n⚠️  Strategy 4 data not available")

# ============================================================================
# SUMMARY
# ============================================================================
print("\n" + "=" * 80)
print("✅ EXPORT SUMMARY")
print("=" * 80)
print(f"\n📂 All files saved to: {export_base_dir}")
print(f"\n📁 Folder structure:")
print(f"   ├── strategy1/")
print(f"   │   └── strategy1_data.csv")
print(f"   ├── strategy2/")
print(f"   │   └── strategy2_data.csv")
print(f"   ├── strategy3/")
print(f"   │   └── strategy3_data.csv")
print(f"   └── strategy4/")
print(f"       └── strategy4_data.csv")

if 'elapsed_s1' in locals() and 'elapsed_s2' in locals() and 'elapsed_s3' in locals() and 'elapsed_s4' in locals():
    total_time = elapsed_s1 + elapsed_s2 + elapsed_s3 + elapsed_s4
    print(f"\n⏱️  Total processing time: {total_time:.2f}s")

print("\n" + "=" * 80)
print("🎉 EXPORT COMPLETE!")
print("=" * 80)

## 🎯 Summary

**Accomplished:**
- ✅ Strategy 1: Format-preserving pseudonymization (SSN with dashes)
- ✅ Strategy 2: Random character pseudonymization (unpredictable output)
- ✅ Strategy 3: Fixed-length pseudonymization (uniform output)
- ✅ Strategy 4: Risk-based conditional pseudonymization (k-anonymity)
- ✅ Privacy metrics calculation (k-anonymity analysis)
- ✅ Performance comparison across strategies
- ✅ Multi-stage pipeline demonstration

**Key takeaways:**
- Format preservation maintains structural integrity while protecting data
- Random pseudonymization provides maximum unpredictability
- Fixed-length pseudonymization standardizes output format
- Risk-based pseudonymization optimizes privacy-utility trade-off
- Different strategies suit different use cases and requirements

**Strategy recommendations:**
- **Use Strategy 1** when: Format structure must be maintained (SSN, phone numbers, IDs)
- **Use Random Pseudonymization** when: Maximum unpredictability is required for security
- **Use Strategy 3** when: Standardized output format is needed across all records
- **Use Risk-Based** when: Balancing privacy protection with data utility is critical

**Next steps:**
- Experiment with different pseudonymize characters and patterns
- Try your own sensitive datasets
- Combine strategies for optimal results
- Deploy to production environment
- Integrate with other anonymization operations

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🐙 [GitHub Repository](https://github.com/DGT-Network/PAMOLA)